In [1]:
%pip install ultralytics opencv-python numpy matplotlib

In [2]:
#nice
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths to seismic images and fault masks
seismic_dir = r'C:\Users\USER\Downloads\try\dataset\raw\seismics1'
mask_dir = r'C:\Users\USER\Downloads\try\dataset\raw\faults'

# Extract IDs from filenames and ignore prefixes
seismic_files = set(f.split('-')[1].split('.')[0] for f in os.listdir(seismic_dir) if f.startswith('seismic-') and f.endswith('.png'))
mask_files = set(f.split('-')[1].split('.')[0] for f in os.listdir(mask_dir) if f.startswith('fault-') and f.endswith('.png'))

# Find common IDs
common_ids = seismic_files.intersection(mask_files)

print(f"Total aligned files: {len(common_ids)}")

# Save the aligned filenames with full paths
aligned_seismic_files = [f"seismic-{id}.png" for id in common_ids]
aligned_fault_files = [f"fault-{id}.png" for id in common_ids]

# Print a few examples to verify
print("Sample aligned files:")
print("Seismic:", aligned_seismic_files[:5])
print("Fault Masks:", aligned_fault_files[:5])

# Create output directories for aligned files
output_seismic_dir = 'dataset/images/aligned'
output_mask_dir = 'dataset/masks/aligned'
os.makedirs(output_seismic_dir, exist_ok=True)
os.makedirs(output_mask_dir, exist_ok=True)

# Copy aligned files to the output directories
for file in common_ids:
    shutil.copy(os.path.join(seismic_dir, f"seismic-{file}.png"), os.path.join(output_seismic_dir, f"seismic-{file}.png"))
    shutil.copy(os.path.join(mask_dir, f"fault-{file}.png"), os.path.join(output_mask_dir, f"fault-{file}.png"))

# Split aligned files into training and validation sets
train_files, val_files = train_test_split(aligned_seismic_files, test_size=0.2, random_state=42)

# Paths to output directories for train and validation sets
train_seismic_dir = 'dataset/images/train'
val_seismic_dir = 'dataset/images/val'
train_mask_dir = 'dataset/masks/train'
val_mask_dir = 'dataset/masks/val'

os.makedirs(train_seismic_dir, exist_ok=True)
os.makedirs(val_seismic_dir, exist_ok=True)
os.makedirs(train_mask_dir, exist_ok=True)
os.makedirs(val_mask_dir, exist_ok=True)

# Move files to train and val directories
for file in train_files:
    shutil.move(os.path.join(output_seismic_dir, file), os.path.join(train_seismic_dir, file))
    shutil.move(os.path.join(output_mask_dir, file.replace("seismic", "fault")), os.path.join(train_mask_dir, file.replace("seismic", "fault")))

for file in val_files:
    shutil.move(os.path.join(output_seismic_dir, file), os.path.join(val_seismic_dir, file))
    shutil.move(os.path.join(output_mask_dir, file.replace("seismic", "fault")), os.path.join(val_mask_dir, file.replace("seismic", "fault")))

print(f"Moved {len(train_files)} files to train and {len(val_files)} files to validation directories.")


Total aligned files: 362
Sample aligned files:
Seismic: ['seismic-1167.png', 'seismic-1005.png', 'seismic-1050.png', 'seismic-1179.png', 'seismic-1126.png']
Fault Masks: ['fault-1167.png', 'fault-1005.png', 'fault-1050.png', 'fault-1179.png', 'fault-1126.png']
Moved 289 files to train and 73 files to validation directories.


In [3]:
#nice

import os
import cv2
import numpy as np
from tqdm import tqdm

# Directories
train_seismic_dir = 'dataset/images/train'
train_mask_dir = 'dataset/masks/train'
val_seismic_dir = 'dataset/images/val'
val_mask_dir = 'dataset/masks/val'

overlay_train_dir = 'dataset/overlays/train'
overlay_val_dir = 'dataset/overlays/val'

# Create directories for overlays
os.makedirs(overlay_train_dir, exist_ok=True)
os.makedirs(overlay_val_dir, exist_ok=True)

# Function to enhance image contrast
def enhance_image(image, alpha=1.3, beta=20):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# Function to create overlay images with natural (grayscale) mask
def create_overlay(seismic_image, mask_image, alpha=0.6):
    # Ensure the mask is 3-channel to blend with the seismic image
    if len(mask_image.shape) == 2:
        mask_image = cv2.cvtColor(mask_image, cv2.COLOR_GRAY2BGR)

    # Blend the enhanced seismic image and the grayscale mask
    enhanced_seismic = enhance_image(seismic_image)
    overlay = cv2.addWeighted(enhanced_seismic, alpha, mask_image, 1 - alpha, 0)
    return overlay

# Process training set
print("Creating overlays for training set...")
for filename in tqdm(os.listdir(train_seismic_dir)):
    seismic_path = os.path.join(train_seismic_dir, filename)
    mask_path = os.path.join(train_mask_dir, filename.replace("seismic", "fault"))

    if os.path.exists(seismic_path) and os.path.exists(mask_path):
        seismic_image = cv2.imread(seismic_path)
        mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        overlay_image = create_overlay(seismic_image, mask_image)
        cv2.imwrite(os.path.join(overlay_train_dir, filename), overlay_image)

# Process validation set
print("Creating overlays for validation set...")
for filename in tqdm(os.listdir(val_seismic_dir)):
    seismic_path = os.path.join(val_seismic_dir, filename)
    mask_path = os.path.join(val_mask_dir, filename.replace("seismic", "fault"))

    if os.path.exists(seismic_path) and os.path.exists(mask_path):
        seismic_image = cv2.imread(seismic_path)
        mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        overlay_image = create_overlay(seismic_image, mask_image)
        cv2.imwrite(os.path.join(overlay_val_dir, filename), overlay_image)

print("Overlay images created successfully.")


Creating overlays for training set...


100%|██████████| 360/360 [07:04<00:00,  1.18s/it]


Creating overlays for validation set...


100%|██████████| 181/181 [03:43<00:00,  1.23s/it]

Overlay images created successfully.


In [4]:
import cv2
import numpy as np
import random

# Preprocessing functions

# 1. Random crop
def random_crop(image, mask, crop_size=(224, 224)):
    h, w = image.shape[:2]
    crop_h, crop_w = crop_size
    top = random.randint(0, h - crop_h)
    left = random.randint(0, w - crop_w)
    cropped_image = image[top:top+crop_h, left:left+crop_w]
    cropped_mask = mask[top:top+crop_h, left:left+crop_w]
    return cropped_image, cropped_mask

# 2. Resize image
def resize_image(image, size=(640, 640)):
    return cv2.resize(image, size)

# 3. Enhance image (CLAHE)
def enhance_image(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

# 4. Normalize image
def normalize_image(image):
    return image / 255.0

# 5. Adjust contrast
def adjust_contrast(image, alpha=1.5, beta=0):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# 6. Thicken fault mask using dilation
def thicken_fault(mask, kernel_size=(5, 5)):
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.dilate(mask, kernel, iterations=1)

# Apply all preprocessing steps to a given image and mask
def preprocess_image_and_mask(image, mask):
    # Random crop
    image, mask = random_crop(image, mask)
    
    # Resize image and mask
    image = resize_image(image)
    mask = resize_image(mask)

    # Randomly apply CLAHE enhancement to image
    if random.random() > 0.5:
        image = enhance_image(image)

    # Random contrast adjustment
    if random.random() > 0.5:
        image = adjust_contrast(image)

    # Normalize image
    image = normalize_image(image)

    # Randomly thicken fault mask
    if random.random() > 0.5:
        mask = thicken_fault(mask)

    return image, mask


In [5]:
# import cv2
# import numpy as np
# import random
# from scipy.ndimage import gaussian_filter, map_coordinates


# # 1. Random Crop
# def random_crop(image, mask, crop_size=(224, 224)):
#     h, w = image.shape[:2]
#     crop_h, crop_w = crop_size
#     top = random.randint(0, h - crop_h)
#     left = random.randint(0, w - crop_w)
#     cropped_image = image[top:top+crop_h, left:left+crop_w]
#     cropped_mask = mask[top:top+crop_h, left:left+crop_w]
#     return cropped_image, cropped_mask


# # 2. Resize Image
# def resize_image(image, size=(640, 640)):
#     return cv2.resize(image, size)


# # 3. Enhance Image (CLAHE)
# def enhance_image(image):
#     lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
#     l, a, b = cv2.split(lab)
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
#     cl = clahe.apply(l)
#     limg = cv2.merge((cl, a, b))
#     return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


# # 4. Normalize Image
# def normalize_image(image):
#     return image / 255.0


# # 5. Adjust Contrast
# def adjust_contrast(image, alpha=1.5, beta=0):
#     return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


# # 6. Thicken Fault Mask (Dilation)
# def thicken_fault(mask, kernel_size=(5, 5)):
#     kernel = np.ones(kernel_size, np.uint8)
#     return cv2.dilate(mask, kernel, iterations=1)


# # 7. Random Rotation
# def random_rotation(image, mask):
#     angle = random.choice([0, 90, 180, 270])
#     if angle == 0:
#         return image, mask
#     rotated_image = cv2.rotate(image, getattr(cv2, f'ROTATE_{angle}_CLOCKWISE'))
#     rotated_mask = cv2.rotate(mask, getattr(cv2, f'ROTATE_{angle}_CLOCKWISE'))
#     return rotated_image, rotated_mask


# # 8. Elastic Transformations
# def elastic_transform(image, mask, alpha=34, sigma=4):
#     random_state = np.random.RandomState(None)
#     shape = image.shape[:2]

#     # Generate random displacement fields
#     dx = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma) * alpha
#     dy = gaussian_filter((random_state.rand(*shape) * 2 - 1), sigma) * alpha

#     # Generate meshgrid of indices
#     x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
#     indices_x = np.clip(x + dx, 0, shape[1] - 1)
#     indices_y = np.clip(y + dy, 0, shape[0] - 1)

#     # Process image
#     if len(image.shape) == 3:  # Color image
#         channels = []
#         for i in range(image.shape[2]):  # Apply transformation per channel
#             channel = map_coordinates(image[:, :, i], [indices_y, indices_x], order=1, mode='reflect')
#             channels.append(channel)
#         warped_image = np.stack(channels, axis=-1)
#     else:  # Grayscale image
#         warped_image = map_coordinates(image, [indices_y, indices_x], order=1, mode='reflect')

#     # Process mask
#     warped_mask = map_coordinates(mask, [indices_y, indices_x], order=1, mode='reflect')

#     return warped_image.astype(np.uint8), warped_mask.astype(np.uint8)



# # 9. Add Random Noise
# def add_random_noise(image, noise_factor=0.05):
#     noise = np.random.randn(*image.shape) * noise_factor
#     noisy_image = np.clip(image + noise, 0, 1)
#     return (noisy_image * 255).astype(np.uint8)


# # 10. Edge Enhancement for Faults
# def enhance_fault_edges(image):
#     gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#     edges = cv2.Canny(gray, 100, 200)
#     return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)  # To keep dimensions consistent


# # 11. Smooth Mask Edges
# def smooth_mask(mask, kernel_size=5):
#     return cv2.GaussianBlur(mask, (kernel_size, kernel_size), 0)


# # Preprocessing Pipeline
# def preprocess_image_and_mask(image, mask):
#     # Random crop
#     image, mask = random_crop(image, mask)
   
#     # Resize image and mask
#     image = resize_image(image)
#     mask = resize_image(mask)

#     # Random augmentations
#     if random.random() > 0.5:
#         image = enhance_image(image)

#     if random.random() > 0.5:
#         image = adjust_contrast(image)

#     if random.random() > 0.5:
#         image = add_random_noise(image)

#     if random.random() > 0.5:
#         image, mask = random_rotation(image, mask)

#     if random.random() > 0.5:
#         image, mask = elastic_transform(image, mask)

#     if random.random() > 0.5:
#         image = enhance_fault_edges(image)

#     # Fault-specific processing
#     if random.random() > 0.5:
#         mask = thicken_fault(mask)

#     if random.random() > 0.5:
#         mask = smooth_mask(mask)

#     # Normalize image
#     image = normalize_image(image)

#     return image, mask


In [6]:
import os
from tqdm import tqdm

# Load and preprocess the dataset (training and validation)
def load_and_preprocess_data(image_dir, mask_dir):
    image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.png')]
    mask_paths = [os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith('.png')]

    data = []
    
    # Ensure that image and mask filenames match (seismic-1000.png -> fault-1000.png)
    for img_path in tqdm(image_paths, total=len(image_paths)):
        filename = os.path.basename(img_path)  # e.g., seismic-1000.png
        mask_path = os.path.join(mask_dir, 'fault-' + filename.split('-')[1])  # e.g., fault-1000.png
        
        if os.path.exists(mask_path):
            # Read the image and mask
            image = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  # Load as grayscale

            # Apply preprocessing
            image, mask = preprocess_image_and_mask(image, mask)

            # Store the processed data
            data.append((image, mask))
        else:
            print(f"Warning: No mask found for {filename}")
    
    return data

# Load and preprocess training data
train_image_dir = 'dataset/images/train'
train_mask_dir = 'dataset/masks/train'
train_data = load_and_preprocess_data(train_image_dir, train_mask_dir)

# Load and preprocess validation data (no augmentation)
val_image_dir = 'dataset/images/val'
val_mask_dir = 'dataset/masks/val'
val_data = load_and_preprocess_data(val_image_dir, val_mask_dir)


100%|██████████| 181/181 [01:48<00:00,  1.67it/s]


In [7]:
import cv2
import os

def create_yolo_segmentation_labels_from_directory(mask_dir, output_dir, class_id=0):
    """
    Creates YOLO segmentation labels for all mask images in a directory.
    
    Args:
        mask_dir (str): Path to the directory containing mask images.
        output_dir (str): Path to the directory where label files will be saved.
        class_id (int): Class ID to assign to all objects.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process each mask image in the directory
    for mask_filename in os.listdir(mask_dir):
        # Construct full file path
        mask_path = os.path.join(mask_dir, mask_filename)
        
        # Check if the file is an image
        if not mask_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            print(f"Skipping non-image file: {mask_filename}")
            continue
        
        # Load the mask image (assume black regions represent the object)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"Could not read mask: {mask_filename}")
            continue
        
        height, width = mask.shape

        # Invert the mask (black -> white, white -> black)
        inverted_mask = cv2.bitwise_not(mask)

        # Find contours in the inverted mask
        contours, _ = cv2.findContours(inverted_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Prepare the label data
        label_data = []
        for contour in contours:
            # Normalize polygon points
            polygon_points = []
            for point in contour:
                px, py = point[0]
                polygon_points.append(f"{px / width} {py / height}")  # Use whitespace instead of comma

            # Combine into one line
            label_line = f"{class_id} " + " ".join(polygon_points)
            label_data.append(label_line)

        # Write to output file
        label_filename = mask_filename.replace("fault", "seismic").replace(".png", ".txt")
        label_filepath = os.path.join(output_dir, label_filename)
        with open(label_filepath, "w") as f:
            f.write("\n".join(label_data))

        print(f"Label created for: {mask_filename} -> {label_filepath}")

In [8]:

# Convert training masks to YOLO format
create_yolo_segmentation_labels_from_directory(
    mask_dir='dataset/masks/train',
    output_dir='dataset/labels/train',
    class_id=0  # Fault class
)

# Convert validation masks to YOLO format
create_yolo_segmentation_labels_from_directory(
    mask_dir='dataset/masks/val',
    output_dir='dataset/labels/val',
    class_id=0  # Fault class
)


Label created for: fault-1000.png -> dataset/labels/train\seismic-1000.txt
Label created for: fault-1001.png -> dataset/labels/train\seismic-1001.txt
Label created for: fault-1002.png -> dataset/labels/train\seismic-1002.txt
Label created for: fault-1003.png -> dataset/labels/train\seismic-1003.txt
Label created for: fault-1004.png -> dataset/labels/train\seismic-1004.txt
Label created for: fault-1005.png -> dataset/labels/train\seismic-1005.txt
Label created for: fault-1006.png -> dataset/labels/train\seismic-1006.txt
Label created for: fault-1007.png -> dataset/labels/train\seismic-1007.txt
Label created for: fault-1008.png -> dataset/labels/train\seismic-1008.txt
Label created for: fault-1009.png -> dataset/labels/train\seismic-1009.txt
Label created for: fault-1010.png -> dataset/labels/train\seismic-1010.txt
Label created for: fault-1011.png -> dataset/labels/train\seismic-1011.txt
Label created for: fault-1012.png -> dataset/labels/train\seismic-1012.txt
Label created for: fault-

In [1]:
from ultralytics import YOLO

# Load a pre-trained YOLO model (YOLOv8n for segmentation)
model = YOLO('yolov8n-seg.pt')  # Use YOLOv8n for segmentation

# Train the model on your dataset
model.train(
    data='seismic_data.yaml',  # Configuration file
    epochs=50,  # Number of epochs
    imgsz=768,  # Image size
    batch=8,    # Batch size
    show_boxes=False,
    name='seismic-fault-detection'  # Experiment name
)

New https://pypi.org/project/ultralytics/8.3.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.47  Python-3.12.0 torch-2.5.1+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
engine\trainer: task=segment, mode=train, model=yolov8n-seg.pt, data=seismic_data.yaml, epochs=50, time=None, patience=100, batch=8, imgsz=768, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=seismic-fault-detection14, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=

train: Scanning C:\Users\USER\Downloads\try\dataset\labels\train.cache... 360 images, 0 backgrounds, 0 corrupt: 100%|██████████| 360/360 [00:00<?, ?it/s]
val: Scanning C:\Users\USER\Downloads\try\dataset\labels\val.cache... 181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 181/181 [00:00<?, ?it/s]


Plotting labels to runs\segment\seismic-fault-detection14\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 768 train, 768 val
Using 0 dataloader workers
Logging results to runs\segment\seismic-fault-detection14
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


       1/50         0G      3.456      3.563      5.917      1.887         35        768: 100%|██████████| 45/45 [07:39<00:00, 10.21s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [01:26<00:00,  7.17s/it]

                   all        181        880     0.0028      0.173    0.00462    0.00116   0.000921     0.0568    0.00123   0.000226


...

                   all        181        880      0.339      0.467      0.297      0.164      0.166      0.278     0.0967     0.0255



50 epochs completed in 7.019 hours.
Optimizer stripped from runs\segment\seismic-fault-detection14\weights\last.pt, 6.8MB
Optimizer stripped from runs\segment\seismic-fault-detection14\weights\best.pt, 6.8MB

Validating runs\segment\seismic-fault-detection14\weights\best.pt...
Ultralytics 8.3.47  Python-3.12.0 torch-2.5.1+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
YOLOv8n-seg summary (fused): 195 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [01:06<00:00,  5.51s/it]


                   all        181        880      0.339      0.466      0.301      0.164      0.169      0.274     0.0993     0.0262
Speed: 2.0ms preprocess, 115.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs\segment\seismic-fault-detection14


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000022E03662B70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.0410

In [2]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Path to the seismic images and YOLO model
image_paths = [
    r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1184.png',  # Path to the first uploaded image
    # r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png',  # Path to the second uploaded image
    # r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png',  # Path to the third uploaded image
]

# Load the trained YOLO model
model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection14\weights\best.pt')

# Create output directory if it doesn't exist
output_dir = r'C:\Users\USER\Downloads\try\dataset\pred_overlay'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Function to process results and overlay masks
def process_and_overlay(image, results):
    if len(results) > 0 and hasattr(results[0], 'masks') and results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()  # Extract mask data
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)  # Initialize an empty mask

        for mask in masks:
            # Resize mask to match the image size
            mask_resized = cv2.resize((mask * 255).astype(np.uint8), (image.shape[1], image.shape[0]))
            combined_mask = cv2.bitwise_or(combined_mask, mask_resized)  # Combine masks

        # Convert mask to 3 channels for overlay
        fault_mask_bgr = cv2.cvtColor(combined_mask, cv2.COLOR_GRAY2BGR)
        # Add the overlay
        overlay = cv2.addWeighted(image, 0.8, fault_mask_bgr, 0.5, 0)
    else:
        print("No masks found in results.")
        overlay = image

    return overlay

# Loop through the uploaded images
for image_path in image_paths:
    # Load the seismic image
    image = cv2.imread(image_path)  # Read the image using OpenCV
    if image is None:
        print(f"Failed to load image: {image_path}")
        continue

    # Perform object detection with YOLO
    results = model(image)  # YOLO inference

    # Process results and create overlay
    overlay_image = process_and_overlay(image, results)

    # Check if any objects are detected
    if len(results) > 0 and hasattr(results[0], 'boxes') and results[0].boxes is not None:
        # Extract bounding boxes, labels, and confidence scores
        boxes = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
        scores = results[0].boxes.conf.cpu().numpy()  # Confidence scores
        labels = results[0].boxes.cls.cpu().numpy()  # Class labels

        print(f"Detected {len(boxes)} faults in {image_path}")

        # Draw bounding boxes and labels on the overlay image
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box  # Coordinates of the bounding box
            score = scores[i]  # Confidence score
            label = labels[i]  # Class label (e.g., "fault")

            # Draw the bounding box on the image
            color = (0, 255, 255)  # Green color for the box
            thickness = 6  # Thickness of the box line
            cv2.rectangle(overlay_image, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)

            # Add the label and confidence score
            cv2.putText(overlay_image, f'{label} ({score:.2f})', (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
    else:
        print(f"No faults detected in {image_path}")

    # Save the processed image with overlay in the output directory
    output_image_path = os.path.join(output_dir, os.path.basename(image_path))
    cv2.imwrite(output_image_path, overlay_image)

    # Display the image for visual confirmation
    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(overlay_image, cv2.COLOR_BGR2RGB))
    plt.title(f"Processed Image: {os.path.basename(image_path)}")
    plt.axis('off')
    plt.show()

print(f"Processed images with overlays saved in: {output_dir}")



0: 512x768 3 faults, 201.1ms
Speed: 7.9ms preprocess, 201.1ms inference, 14.5ms postprocess per image at shape (1, 3, 512, 768)
Detected 3 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1184.png


<Figure size 1000x500 with 1 Axes>

Processed images with overlays saved in: C:\Users\USER\Downloads\try\dataset\pred_overlay


In [ ]:
# import cv2
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# from ultralytics import YOLO

# # Path to the seismic images and YOLO model
# image_paths = [
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1008.png',  # Path to the first uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png',  # Path to the second uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png',  # Path to the third uploaded image
# ]

# # Load the trained YOLO model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection11\weights\best.pt')

# # Create output directory if it doesn't exist
# output_dir = r'C:\Users\USER\Downloads\try\dataset'
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# # Loop through the uploaded images
# for image_path in image_paths:
#     # Load the seismic image
#     image = cv2.imread(image_path)  # Read the image using OpenCV
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB for correct visualization

#     # Perform object detection with YOLO
#     results = model(image)  # YOLO inference

#     # Check if any objects are detected
#     if len(results) > 0 and hasattr(results[0], 'boxes') and results[0].boxes is not None:
#         # Extract bounding boxes, labels, and confidence scores
#         boxes = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
#         scores = results[0].boxes.conf.cpu().numpy()  # Confidence scores
#         labels = results[0].boxes.cls.cpu().numpy()  # Class labels

#         print(f"Detected {len(boxes)} faults in {image_path}")
#     else:
#         print(f"No faults detected in {image_path}")
#         boxes = []
#         scores = []
#         labels = []

#     # Visualize the seismic image and the detected fault regions
#     for i, box in enumerate(boxes):
#         x1, y1, x2, y2 = box  # Coordinates of the bounding box
#         score = scores[i]  # Confidence score
#         label = labels[i]  # Class label (e.g., "fault")

#         # Draw the bounding box on the image
#         color = (0, 255, 0)  # Green color for the box
#         thickness = 6  # Thickness of the box line
#         cv2.rectangle(image_rgb, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)
        

#         # Add the label and confidence score
#         cv2.putText(image_rgb, f'{label} ({score:.2f})', (int(x1), int(y1)-10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

#     # Convert image back to BGR for OpenCV compatibility if needed
#     image_rgb_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

#     # Save the image with bounding boxes in the output folder
#     output_image_path = os.path.join(output_dir, os.path.basename(image_path))
#     cv2.imwrite(output_image_path, image_rgb_bgr)

#     # Optional: Display the image for visual confirmation
#     plt.figure(figsize=(10, 5))
#     plt.imshow(image_rgb)
#     plt.title(f"Processed Image: {os.path.basename(image_path)}")
#     plt.axis('off')
#     plt.show()

# print(f"Processed images saved in: {output_dir}")



0: 512x768 1 fault, 250.2ms
Speed: 41.2ms preprocess, 250.2ms inference, 22.1ms postprocess per image at shape (1, 3, 512, 768)
Detected 1 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1008.png


<Figure size 1000x500 with 1 Axes>


0: 512x768 3 faults, 120.4ms
Speed: 4.3ms preprocess, 120.4ms inference, 9.4ms postprocess per image at shape (1, 3, 512, 768)
Detected 3 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png


<Figure size 1000x500 with 1 Axes>


0: 512x768 1 fault, 146.4ms
Speed: 2.5ms preprocess, 146.4ms inference, 7.0ms postprocess per image at shape (1, 3, 512, 768)
Detected 1 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png


<Figure size 1000x500 with 1 Axes>

Processed images saved in: C:\Users\USER\Downloads\try\dataset


In [ ]:
# import cv2
# import numpy as np
# from ultralytics import YOLO

# # Load the trained YOLO model
# model = YOLO('runs/segment/seismic-fault-detection11/weights/best.pt')  # Path to your trained model

# def infer(input_image_path, post_process=True):
#     """
#     Perform inference on an input image and overlay fault detection masks.

#     Args:
#         input_image_path (str): Path to the input image.
#         post_process (bool): Whether to apply post-processing (default: True).

#     Returns:
#         overlay (numpy.ndarray): Image with fault detection masks overlaid.
#     """
#     # Load the input image
#     input_image = cv2.imread(input_image_path)
#     if input_image is None:
#         raise ValueError(f"Unable to load image from path: {input_image_path}")

#     # Run inference on the input image
#     results = model.predict(source=input_image_path, save=False, conf=0.05)
    
#     # Check if masks are available in the results
#     for result in results:
#         if result.masks is not None:
#             # Extract mask data from the Masks object
#             masks = result.masks.data.cpu().numpy()  # Convert to NumPy array
#             combined_mask = np.zeros(input_image.shape[:2], dtype=np.uint8)  # 2D empty mask

#             # Iterate over each mask
#             for mask in masks:
#                 # Convert mask to binary and resize to match the input image size
#                 mask_resized = cv2.resize((mask * 255).astype(np.uint8), (input_image.shape[1], input_image.shape[0]))
#                 combined_mask = cv2.bitwise_or(combined_mask, mask_resized)  # Combine masks

#             # Post-process the combined mask
#             if post_process:
#                 # Invert the mask to make fault regions black
#                 inverted_mask = cv2.bitwise_not(combined_mask)
#                 # Apply dilation to expand the fault regions
#                 kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
#                 dilated_mask = cv2.dilate(inverted_mask, kernel, iterations=1)
#                 # Apply morphological closing to fill gaps
#                 closed_mask = cv2.morphologyEx(dilated_mask, cv2.MORPH_CLOSE, kernel)
#                 pred_mask = cv2.bitwise_not(closed_mask)
#             else:
#                 pred_mask = combined_mask

#             # Convert binary mask to a 3-channel BGR image
#             fault_mask_bgr = cv2.cvtColor(pred_mask, cv2.COLOR_GRAY2BGR)

#             # Overlay the fault mask onto the input image
#             overlay = cv2.addWeighted(input_image, 0.8, fault_mask_bgr, 0.5, 0)
            
#             return overlay

#     print("No masks found in the results.")
#     return None


In [ ]:
# import cv2
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# from ultralytics import YOLO

# # Path to the seismic images and YOLO model
# image_paths = [
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png',  # Path to the first uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png',  # Path to the second uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png',  # Path to the third uploaded image
# ]

# # Load the trained YOLO model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection11\weights\best.pt')

# # Create output directory if it doesn't exist
# output_dir = r'C:\Users\USER\Downloads\try\output_with_overlay'
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# # Define the infer function
# def infer(input_image_path, post_process=True):
#     """
#     Perform inference on an input image and overlay fault detection masks.

#     Args:
#         input_image_path (str): Path to the input image.
#         post_process (bool): Whether to apply post-processing (default: True).

#     Returns:
#         overlay (numpy.ndarray): Image with fault detection masks overlaid.
#     """
#     input_image = cv2.imread(input_image_path)
#     if input_image is None:
#         raise ValueError(f"Unable to load image from path: {input_image_path}")

#     # results = model.predict(source=input_image_path, save=False, conf=0.2)
#     results = model.predict(source=input_image_path, save=False, imgsz=768)

#     for result in results:
#         if result.masks is not None:
#             masks = result.masks.data.cpu().numpy()
#             combined_mask = np.zeros(input_image.shape[:2], dtype=np.uint8)

#             for mask in masks:
#                 mask_resized = cv2.resize((mask * 255).astype(np.uint8), (input_image.shape[1], input_image.shape[0]))
#                 combined_mask = cv2.bitwise_or(combined_mask, mask_resized)

#             if post_process:
#                 inverted_mask = cv2.bitwise_not(combined_mask)
#                 kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
#                 dilated_mask = cv2.dilate(inverted_mask, kernel, iterations=1)
#                 closed_mask = cv2.morphologyEx(dilated_mask, cv2.MORPH_CLOSE, kernel)
#                 pred_mask = cv2.bitwise_not(closed_mask)
#             else:
#                 pred_mask = combined_mask

#             fault_mask_bgr = cv2.cvtColor(pred_mask, cv2.COLOR_GRAY2BGR)
#             overlay = cv2.addWeighted(input_image, 0.8, fault_mask_bgr, 0.5, 0)

#             return overlay

#     print("No masks found in results.")
#     return None

# # Loop through the uploaded images
# for image_path in image_paths:
#     # Perform inference and get overlay result
#     overlay_result = infer(image_path)

#     if overlay_result is not None:
#         # Save the overlay result in the output directory
#         output_image_path = os.path.join(output_dir, os.path.basename(image_path))
#         cv2.imwrite(output_image_path, overlay_result)

#         # Display the image for visual confirmation
#         plt.figure(figsize=(10, 5))
#         plt.imshow(cv2.cvtColor(overlay_result, cv2.COLOR_BGR2RGB))
#         plt.title(f"Processed Image: {os.path.basename(image_path)}")
#         plt.axis('off')
#         plt.show()
#     else:
#         print(f"No faults detected or processing failed for {image_path}.")

# print(f"Processed images with overlays saved in: {output_dir}")


image 1/1 C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png: 512x768 3 faults, 106.2ms
Speed: 0.0ms preprocess, 106.2ms inference, 8.4ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x500 with 1 Axes>


image 1/1 C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png: 512x768 3 faults, 113.2ms
Speed: 0.0ms preprocess, 113.2ms inference, 9.2ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x500 with 1 Axes>


image 1/1 C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png: 512x768 1 fault, 105.3ms
Speed: 8.0ms preprocess, 105.3ms inference, 50.0ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x500 with 1 Axes>

Processed images with overlays saved in: C:\Users\USER\Downloads\try\output_with_overlay


In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np

# # Load the trained model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection8\weights\best.pt')

# # Load a test image
# image_path = r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png'
# image = cv2.imread(image_path)

# # Check if the image is loaded correctly
# if image is None:
#     print(f"Error: Unable to load image at {image_path}")
#     exit()

# # Perform inference
# results = model.predict(source=image, save=False)

# # Visualize the results
# for result in results:
#     if result.masks is not None:
#         mask = result.masks.data.cpu().numpy()[0]  # Extract the first mask
#         mask_resized = cv2.resize((mask * 255).astype(np.uint8), (image.shape[1], image.shape[0]))  # Resize to original image shape
#         cv2.imshow('Fault Mask', mask_resized)  # Display the mask
#         cv2.waitKey(0)  # Wait for a key press to close the window
#         cv2.destroyAllWindows()  # Close the window after key press


0: 512x768 3 faults, 120.6ms
Speed: 0.0ms preprocess, 120.6ms inference, 0.0ms postprocess per image at shape (1, 3, 512, 768)


In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np

# # Load the trained model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection8\weights\best.pt')

# # Load a test image
# image_path = r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1068.png'
# image = cv2.imread(image_path)

# # Check if the image is loaded correctly
# if image is None:
#     print(f"Error: Unable to load image at {image_path}")
#     exit()
# else:
#     print("Image loaded successfully!")
#     cv2.imshow('Original Image', image)  # Display the original image
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

# # Perform inference
# results = model.predict(source=image, save=False)

# # Visualize the results
# for result in results:
#     if result.masks is not None:
#         print("Mask detected!")
#         mask = result.masks.data.cpu().numpy()[0]  # Extract the first mask
#         mask_resized = cv2.resize((mask * 255).astype(np.uint8), (image.shape[1], image.shape[0]))  # Resize to original image shape
#         cv2.imshow('Fault Mask', mask_resized)  # Display the mask
#         cv2.waitKey(0)  # Wait for a key press to close the window
#         cv2.destroyAllWindows()
#     else:
#         print("No mask detected.")

Image loaded successfully!

0: 512x768 3 faults, 191.4ms
Speed: 9.1ms preprocess, 191.4ms inference, 6.7ms postprocess per image at shape (1, 3, 512, 768)
Mask detected!


In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# # Load the trained model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection8\weights\best.pt')

# # Load a test image
# image_path = r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png'
# image = cv2.imread(image_path)

# # Check if the image is loaded correctly
# if image is None:
#     print(f"Error: Unable to load image at {image_path}")
#     exit()
# else:
#     print("Image loaded successfully!")

# # Perform inference
# results = model.predict(source=image, save=False)

# # Check if the results contain masks
# if not results or not hasattr(results[0], 'masks') or results[0].masks is None:
#     print("No masks found in results")
# else:
#     print(f"Found {len(results)} results, with mask data")

# # Prepare to plot the original image and the detected mask
# fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# # Show the original image
# axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for proper display in matplotlib
# axes[0].set_title('Original Image')
# axes[0].axis('off')  # Hide axes

# # Visualize the mask
# mask_displayed = False
# for result in results:
#     if result.masks is not None:
#         # Extract the first mask
#         mask = result.masks.data.cpu().numpy()[0]
        
#         # Apply thresholding to ensure the mask is clearly visible
#         mask[mask < 0.5] = 0  # Set values below 0.5 to 0 (remove faint areas)
#         mask[mask >= 0.5] = 1  # Set values above or equal to 0.5 to 1 (highlight fault areas)
        
#         # Resize to match the original image shape
#         mask_resized = cv2.resize(mask.astype(np.uint8) * 255, (image.shape[1], image.shape[0]))  # Resize to original image shape
        
#         # Show the mask in grayscale
#         axes[1].imshow(mask_resized, cmap='gray')
#         axes[1].set_title('Detected Fault Mask')
#         axes[1].axis('off')  # Hide axes
#         mask_displayed = True

# # If no mask is found, inform the user
# if not mask_displayed:
#     axes[1].text(0.5, 0.5, 'No fault mask detected', ha='center', va='center', fontsize=12, color='red')

# plt.tight_layout()
# plt.show()


Image loaded successfully!

0: 512x768 3 faults, 114.6ms
Speed: 8.1ms preprocess, 114.6ms inference, 2.4ms postprocess per image at shape (1, 3, 512, 768)
Found 1 results, with mask data


<Figure size 1200x600 with 2 Axes>

In [ ]:
# from ultralytics import YOLO
# import cv2
# import matplotlib.pyplot as plt

# # Load the trained YOLO model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection8\weights\best.pt')

# # Function to display the result
# def show_results(image, results):
#     # If results exist, we plot them
#     for result in results:
#         if result.masks is not None:
#             mask = result.masks.data.cpu().numpy()[0]
#             mask_resized = cv2.resize((mask * 255).astype(np.uint8), (image.shape[1], image.shape[0]))
            
#             # Display the image and mask
#             plt.figure(figsize=(10, 10))
#             plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))  # Convert BGR to RGB for proper display
#             plt.imshow(mask_resized, cmap='jet', alpha=0.5)  # Overlay mask on the image
#             plt.show()

# # Loop through the uploaded images
# image_paths = [
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png',  # Path to the first uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png',  # Path to the first uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png',  # Path to the first uploaded image
# ]

# for image_path in image_paths:
#     # Load each image
#     image = cv2.imread(image_path)

#     # Perform inference
#     results = model.predict(source=image, save=False)

#     # Show the result
#     show_results(image, results)



0: 512x768 3 faults, 186.3ms
Speed: 18.9ms preprocess, 186.3ms inference, 16.5ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x1000 with 1 Axes>


0: 512x768 3 faults, 113.1ms
Speed: 6.7ms preprocess, 113.1ms inference, 5.2ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x1000 with 1 Axes>


0: 512x768 3 faults, 127.2ms
Speed: 7.1ms preprocess, 127.2ms inference, 9.3ms postprocess per image at shape (1, 3, 512, 768)


<Figure size 1000x1000 with 1 Axes>

In [ ]:
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# from ultralytics import YOLO

# # Path to the seismic images and YOLO model
# image_paths = [
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png',  # Path to the first uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png',  # Path to the second uploaded image
#     r'C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png',  # Path to the third uploaded image
# ]

# # Load the trained YOLO model
# model = YOLO(r'C:\Users\USER\Downloads\try\runs\segment\seismic-fault-detection8\weights\best.pt')

# # Loop through the uploaded images
# for image_path in image_paths:
#     # Load the seismic image
#     image = cv2.imread(image_path)  # Read the image using OpenCV
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB for correct visualization

#     # Perform object detection with YOLO
#     results = model(image)  # YOLO inference

#     # Check if any objects are detected
#     if len(results) > 0 and hasattr(results[0], 'boxes') and results[0].boxes is not None:
#         # Extract bounding boxes, labels, and confidence scores
#         boxes = results[0].boxes.xyxy.cpu().numpy()  # Bounding boxes
#         scores = results[0].boxes.conf.cpu().numpy()  # Confidence scores
#         labels = results[0].boxes.cls.cpu().numpy()  # Class labels

#         print(f"Detected {len(boxes)} faults in {image_path}")
#     else:
#         print(f"No faults detected in {image_path}")
#         boxes = []
#         scores = []
#         labels = []

#     # Visualize the seismic image and the detected fault regions
#     plt.figure(figsize=(10, 5))

#     # Display the original seismic image
#     plt.subplot(1, 2, 1)
#     plt.imshow(image_rgb)
#     plt.title("Seismic Image")

#     # Draw bounding boxes for detected faults
#     for i, box in enumerate(boxes):
#         x1, y1, x2, y2 = box  # Coordinates of the bounding box
#         score = scores[i]  # Confidence score
#         label = labels[i]  # Class label (e.g., "fault")

#         # Draw the bounding box on the image
#         color = (0, 255, 0)  # Green color for the box
#         thickness = 2  # Thickness of the box line
#         cv2.rectangle(image_rgb, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)

#         # Add the label and confidence score
#         cv2.putText(image_rgb, f'{label} ({score:.2f})', (int(x1), int(y1)-10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

#     # Convert image back to BGR for OpenCV compatibility if needed
#     image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

#     plt.axis('off')

#     # Display the predicted fault mask (just the bounding boxes)
#     plt.subplot(1, 2, 2)
#     plt.imshow(image_rgb)
#     plt.title("Detected Faults")
#     plt.axis('off')

#     plt.tight_layout()
#     plt.show()



0: 512x768 3 faults, 217.6ms
Speed: 40.2ms preprocess, 217.6ms inference, 22.3ms postprocess per image at shape (1, 3, 512, 768)
Detected 3 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1126.png


<Figure size 1000x500 with 2 Axes>


0: 512x768 3 faults, 138.6ms
Speed: 15.5ms preprocess, 138.6ms inference, 9.6ms postprocess per image at shape (1, 3, 512, 768)
Detected 3 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1129.png


<Figure size 1000x500 with 2 Axes>


0: 512x768 3 faults, 151.8ms
Speed: 4.4ms preprocess, 151.8ms inference, 8.2ms postprocess per image at shape (1, 3, 512, 768)
Detected 3 faults in C:\Users\USER\Downloads\try\dataset\images\val\seismic-1133.png


<Figure size 1000x500 with 2 Axes>